In [ ]:
!pip install -q transformers datasets accelerate scikit-learn safetensors

In [ ]:
import os, re, zipfile, gc
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoConfig,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from huggingface_hub import hf_hub_download

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── CUDA memory fragmentation fix ────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Hyper-parameters ─────────────────────────────────────────────
MODEL_NAME   = "google/muril-base-cased"   # MuRIL for Indic languages
MAX_LEN      = 256
NUM_EPOCHS   = 20
BATCH_SIZE   = 4
EVAL_BATCH   = 8
GRAD_ACCUM   = 2
LR           = 2e-5
WARMUP_EPOCHS = 1.5                       # warmup for ~1.5 epochs
WEIGHT_DECAY = 0.01
FP16         = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── Data paths (change to match your environment) ────────────────
TRAIN_PATH = "mreddit2026/training_data/tulu_train_data.csv"
TEST_PATH  = "mreddit2026/test_data/tulu_test_data.csv"

In [ ]:
def load_muril_for_classification(num_labels: int) -> BertForSequenceClassification:
    """
    Load MuRIL with proper key remapping:
      gamma → weight, beta → bias  (LayerNorm)
    Returns a BertForSequenceClassification ready for fine-tuning.
    """
    config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=num_labels)
    model  = BertForSequenceClassification(config)

    # ── Download checkpoint ──────────────────────────────────────
    try:
        fpath = hf_hub_download(MODEL_NAME, "pytorch_model.bin")
        orig_state = torch.load(fpath, map_location="cpu", weights_only=True)
    except Exception:
        fpath = hf_hub_download(MODEL_NAME, "model.safetensors")
        from safetensors.torch import load_file
        orig_state = load_file(fpath)

    # ── Remap keys ──────────────────────────────────────────────
    new_state = {}
    for k, v in orig_state.items():
        nk = k.replace(".gamma", ".weight").replace(".beta", ".bias")
        new_state[nk] = v

    missing, unexpected = model.load_state_dict(new_state, strict=False)
    print(f"  Missing keys  (classifier head — expected): {missing}")
    print(f"  Unexpected keys (should be empty):          {unexpected}")

    return model

In [ ]:
def light_clean(text: str) -> str:
    text = re.sub(r"<[^>]+>", "", str(text))
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ── Label maps ───────────────────────────────────────────────────
regret_map = {"No Regret": 0, "Action": 1, "Inaction": 2}
regret_rev = {v: k for k, v in regret_map.items()}

domain_map = {
    "Romance friends and parents": 0,
    "Other domains": 1,
    "Education": 2,
    "Career and Finance": 3,
    "Health": 4,
}
domain_rev = {v: k for k, v in domain_map.items()}

# ── Read CSVs ────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

train_df["full_text"] = (
    train_df["title"].fillna("") + " - " + train_df["text"].fillna("")
).apply(light_clean)
test_df["full_text"] = (
    test_df["title"].fillna("") + " - " + test_df["text"].fillna("")
).apply(light_clean)

train_df["regret_label"] = train_df["Regret"].map(regret_map)
train_df["domain_label"] = train_df["Domain"].map(domain_map)

print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")
print(train_df[["Regret", "regret_label"]].value_counts())
print()
print(train_df[["Domain", "domain_label"]].value_counts())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ReDDITDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])


def compute_metrics(pred):
    labels     = pred.label_ids
    predictions = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    preds = predictions.argmax(-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
    }


def make_weighted_trainer(model, train_ds, val_ds, weights_tensor, out_dir):
    """Trainer with class-weighted CrossEntropyLoss + dynamic warmup steps."""

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels  = inputs.pop("labels")
            outputs = model(**inputs)

            device = outputs.logits.device
            loss_fct = nn.CrossEntropyLoss(weight=weights_tensor.to(device))

            num_labels = (
                model.module.config.num_labels
                if hasattr(model, "module")
                else model.config.num_labels
            )
            loss = loss_fct(outputs.logits.view(-1, num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    # ── Compute warmup steps from WARMUP_EPOCHS ─────────────────
    steps_per_epoch = max(1, len(train_ds) // (BATCH_SIZE * GRAD_ACCUM))
    warmup_steps    = int(steps_per_epoch * WARMUP_EPOCHS)

    args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_steps=warmup_steps,
        weight_decay=WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        fp16=FP16,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
    )

    return WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

In [ ]:
tr_txt, val_txt, tr_lab, val_lab = train_test_split(
    train_df["full_text"].tolist(),
    train_df["regret_label"].tolist(),
    test_size=0.1,
    stratify=train_df["regret_label"].tolist(),
    random_state=SEED,
)

train_ds_r = ReDDITDataset(tr_txt, tr_lab)
val_ds_r   = ReDDITDataset(val_txt, val_lab)

w_regret = torch.tensor(
    compute_class_weight(
        "balanced",
        classes=np.unique(train_df["regret_label"]),
        y=train_df["regret_label"],
    ),
    dtype=torch.float32,
).to(DEVICE)

print("Loading MuRIL for Regret classification …")
model_regret = load_muril_for_classification(num_labels=3).to(DEVICE)

trainer_regret = make_weighted_trainer(
    model_regret, train_ds_r, val_ds_r, w_regret, "./results_regret"
)
trainer_regret.train()

In [ ]:
gc.collect()
torch.cuda.empty_cache()

dom_df = train_df.dropna(subset=["domain_label"])

tr_txt_d, val_txt_d, tr_lab_d, val_lab_d = train_test_split(
    dom_df["full_text"].tolist(),
    dom_df["domain_label"].astype(int).tolist(),
    test_size=0.1,
    stratify=dom_df["domain_label"].astype(int).tolist(),
    random_state=SEED,
)

train_ds_d = ReDDITDataset(tr_txt_d, tr_lab_d)
val_ds_d   = ReDDITDataset(val_txt_d, val_lab_d)

w_domain = torch.tensor(
    compute_class_weight(
        "balanced",
        classes=np.unique(dom_df["domain_label"]),
        y=dom_df["domain_label"],
    ),
    dtype=torch.float32,
).to(DEVICE)

print("Loading MuRIL for Domain classification …")
model_domain = load_muril_for_classification(num_labels=5).to(DEVICE)

trainer_domain = make_weighted_trainer(
    model_domain, train_ds_d, val_ds_d, w_domain, "./results_domain"
)
trainer_domain.train()

In [ ]:
test_ds = ReDDITDataset(test_df["full_text"].tolist())

regret_preds = trainer_regret.predict(test_ds).predictions.argmax(-1)
domain_preds = trainer_domain.predict(test_ds).predictions.argmax(-1)

sub_df = pd.DataFrame({
    "id":     test_df["id"],
    "Regret": [regret_rev[p] for p in regret_preds],
    "Domain": [domain_rev[p] for p in domain_preds],
})

sub_df.to_csv("predictions.csv", index=False)

with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("predictions.csv", arcname="predictions.csv")

print("✅  submission.zip ready")
print(sub_df.head())